# 11 — Fine-Tuning OpenAI gpt-4o-mini para Classificação de Tickets

## Objetivo
Fine-tunar **gpt-4o-mini** no Dataset 2 (IT Service Tickets) e comparar com o baseline Gemini few-shot (46.2%).

### Abordagem
- **Treino:** 80% do 20% sample (~7.647 tickets)
- **Validação:** 20% do 20% sample (~1.912 tickets) — mesma distribuição que o teste Gemini
- **Teste:** Mesma amostra de validação para comparação direta
- **Custo estimado:** ~$9 treino (20% do dataset × 3 epochs)

### Mesmos padrões do notebook 07
- Mesmas 8 categorias
- Mesma semente aleatória (42)
- Mesma fração de amostra (20%)
- Checkpoint system para resiliência

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import time
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (classification_report, confusion_matrix,
                            accuracy_score, f1_score)
import warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv
load_dotenv(".env.local")

from openai import OpenAI
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print("OpenAI client configurado")

RANDOM_STATE = 42
SAMPLE_FRACTION = 0.20
TRAIN_FRACTION = 0.80  # 80% treino, 20% validação/teste
MODEL = "gpt-4o-mini-2024-07-18"

VALID_CATEGORIES = ["Access", "Administrative rights", "HR Support", "Hardware",
                    "Internal Project", "Miscellaneous", "Purchase", "Storage"]

SYSTEM_PROMPT = ("You are an IT ticket classifier. Classify the ticket into exactly ONE category: "
                 "Access, Administrative rights, HR Support, Hardware, Internal Project, "
                 "Miscellaneous, Purchase, Storage. Respond with ONLY the category name.")

csv_path = "data/it_service_tickets.csv"
df = pd.read_csv(csv_path)
print(f"Dataset carregado: {len(df)} tickets")
print(f"Categorias: {df['Topic_group'].nunique()}")
print(df["Topic_group"].value_counts().to_string())

## Seção 1: Preparação da Amostra (mesmo split do notebook 07)

In [ ]:
# Reservar os mesmos 40 exemplos few-shot que o notebook 07 usou
# (para não ter overlap entre treino e os exemplos do Gemini)
np.random.seed(RANDOM_STATE)
fewshot_indices = []
for cat in VALID_CATEGORIES:
    cat_data = df[df["Topic_group"] == cat]
    sample_idx = cat_data.sample(n=5, random_state=RANDOM_STATE).index
    fewshot_indices.extend(sample_idx.tolist())

df_remaining = df.drop(index=fewshot_indices)

# Mesma amostra de 20% que o Gemini usou
sample = df_remaining.groupby("Topic_group", group_keys=False).apply(
    lambda x: x.sample(frac=SAMPLE_FRACTION, random_state=RANDOM_STATE)
).reset_index(drop=True)
sample = sample.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print(f"Amostra total: {len(sample)} tickets (20%)")

# Split treino/validação
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(
    sample, test_size=1-TRAIN_FRACTION, random_state=RANDOM_STATE, stratify=sample["Topic_group"]
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Treino: {len(train_df)} tickets")
print(f"Validação/Teste: {len(val_df)} tickets")
print(f"\nDistribuição treino:")
print(train_df["Topic_group"].value_counts().to_string())
print(f"\nDistribuição validação:")
print(val_df["Topic_group"].value_counts().to_string())

## Seção 2: Gerar JSONL para Fine-Tuning

In [ ]:
def row_to_jsonl(text, category):
    """Converte um ticket para o formato JSONL do OpenAI fine-tuning."""
    return json.dumps({
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": str(text)},
            {"role": "assistant", "content": category}
        ]
    })

# Gerar arquivos JSONL
train_path = "data/openai_train.jsonl"
val_path = "data/openai_val.jsonl"

with open(train_path, "w") as f:
    for _, row in train_df.iterrows():
        f.write(row_to_jsonl(row["Document"], row["Topic_group"]) + "\n")

with open(val_path, "w") as f:
    for _, row in val_df.iterrows():
        f.write(row_to_jsonl(row["Document"], row["Topic_group"]) + "\n")

print(f"Treino: {train_path} ({len(train_df)} linhas, {os.path.getsize(train_path)/1024:.1f} KB)")
print(f"Validação: {val_path} ({len(val_df)} linhas, {os.path.getsize(val_path)/1024:.1f} KB)")

# Verificar formato
with open(train_path) as f:
    first = json.loads(f.readline())
print(f"\nExemplo (primeira linha):")
print(json.dumps(first, indent=2)[:500])

## Seção 3: Upload e Fine-Tuning

In [ ]:
# Upload dos arquivos
print("Uploading training file...")
train_file = client.files.create(
    file=open(train_path, "rb"),
    purpose="fine-tune"
)
print(f"  Training file ID: {train_file.id}")

print("Uploading validation file...")
val_file = client.files.create(
    file=open(val_path, "rb"),
    purpose="fine-tune"
)
print(f"  Validation file ID: {val_file.id}")

# Criar job de fine-tuning
print(f"\nCriando fine-tuning job ({MODEL})...")
job = client.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=val_file.id,
    model=MODEL,
    suffix="optiflow-tickets"
)
print(f"  Job ID: {job.id}")
print(f"  Status: {job.status}")
print(f"\nAcompanhe em: https://platform.openai.com/finetune/{job.id}")

## Seção 4: Monitorar Fine-Tuning

In [ ]:
# Auto-polling: aguarda fine-tuning completar (checa a cada 60s)
# Progresso visível no terminal via papermill --log-output

print("=" * 60)
print(f"AGUARDANDO FINE-TUNING — Job: {job.id}")
print(f"Dashboard: https://platform.openai.com/finetune/{job.id}")
print("=" * 60)

FINE_TUNED_MODEL = None
poll_interval = 60  # segundos
max_wait = 3600  # 1 hora max
waited = 0

while waited < max_wait:
    job_status = client.fine_tuning.jobs.retrieve(job.id)
    status = job_status.status
    
    if status == "succeeded":
        FINE_TUNED_MODEL = job_status.fine_tuned_model
        print(f"\n✅ MODELO PRONTO: {FINE_TUNED_MODEL}")
        break
    elif status == "failed":
        print(f"\n❌ FALHOU: {job_status.error}")
        raise RuntimeError(f"Fine-tuning failed: {job_status.error}")
    elif status == "cancelled":
        raise RuntimeError("Fine-tuning was cancelled")
    else:
        # Mostrar último evento
        try:
            events = client.fine_tuning.jobs.list_events(fine_tuning_job_id=job.id, limit=1)
            last_event = events.data[0].message if events.data else "aguardando..."
        except:
            last_event = "aguardando..."
        
        mins = waited // 60
        print(f"  [{mins}min] Status: {status} — {last_event}")
        time.sleep(poll_interval)
        waited += poll_interval

if FINE_TUNED_MODEL is None:
    raise RuntimeError(f"Timeout após {max_wait//60}min. Verifique no dashboard.")

## Seção 5: Avaliação do Modelo Fine-Tuned

⚠️ Só rodar depois que o fine-tuning terminar (Seção 4 mostra `succeeded`)

In [ ]:
# Se voltando de uma sessão anterior, preencha o model ID aqui:
# FINE_TUNED_MODEL = "ft:gpt-4o-mini-2024-07-18:your-org:optiflow-tickets:xxxxx"

print(f"Modelo: {FINE_TUNED_MODEL}")
print(f"Avaliando em {len(val_df)} tickets de validação...")

def normalize_prediction(pred):
    pred = pred.strip().strip(".").strip()
    for cat in VALID_CATEGORIES:
        if pred.lower() == cat.lower():
            return cat
    for cat in VALID_CATEGORIES:
        if cat.lower() in pred.lower():
            return cat
    return pred

# Checkpoint system (mesmo padrão do notebook 07)
BATCH_SIZE = 200
checkpoint_path = "data/openai_ft_checkpoint.csv"

if os.path.exists(checkpoint_path):
    done = pd.read_csv(checkpoint_path)
    start_idx = len(done)
    all_results = done.to_dict("records")
    print(f"Checkpoint encontrado: {start_idx} já processados")
else:
    start_idx = 0
    all_results = []

errors = 0
start_time = time.time()

for i in range(start_idx, len(val_df)):
    row = val_df.iloc[i]
    try:
        response = client.chat.completions.create(
            model=FINE_TUNED_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": str(row["Document"])}
            ],
            max_tokens=10,
            temperature=0
        )
        pred = normalize_prediction(response.choices[0].message.content)
        all_results.append({"true_label": row["Topic_group"], "predicted": pred})
    except Exception as e:
        all_results.append({"true_label": row["Topic_group"], "predicted": "ERROR"})
        errors += 1
        time.sleep(3)
    
    processed = i - start_idx + 1
    if processed % BATCH_SIZE == 0 or i == len(val_df) - 1:
        pd.DataFrame(all_results).to_csv(checkpoint_path, index=False)
        elapsed = time.time() - start_time
        rate = processed / elapsed * 60 if elapsed > 0 else 0
        remaining = (len(val_df) - i - 1) / (rate / 60) if rate > 0 else 0
        print(f"  [{i+1}/{len(val_df)}] {rate:.0f}/min, erros={errors}, "
              f"restante~{remaining/60:.0f}min [checkpoint salvo]")

results_df = pd.DataFrame(all_results)
elapsed = time.time() - start_time
print(f"\nConcluido em {elapsed/60:.1f}min, {errors} erros")

## Seção 6: Métricas e Comparação com Gemini

In [ ]:
ft_eval = results_df[results_df["predicted"] != "ERROR"]
ft_acc = accuracy_score(ft_eval["true_label"], ft_eval["predicted"])
ft_f1_macro = f1_score(ft_eval["true_label"], ft_eval["predicted"], average="macro", zero_division=0)
ft_f1_weighted = f1_score(ft_eval["true_label"], ft_eval["predicted"], average="weighted", zero_division=0)

print("=" * 60)
print("RESULTADOS: gpt-4o-mini FINE-TUNED")
print("=" * 60)
print(f"Acurácia:    {ft_acc:.1%}")
print(f"F1 Macro:    {ft_f1_macro:.3f}")
print(f"F1 Weighted: {ft_f1_weighted:.3f}")
print(f"Erros API:   {len(results_df) - len(ft_eval)}")
print()
print(classification_report(ft_eval["true_label"], ft_eval["predicted"], zero_division=0))

# Comparação com Gemini
print("\n" + "=" * 60)
print("COMPARAÇÃO: Gemini Few-Shot vs OpenAI Fine-Tuned")
print("=" * 60)
print(f"{'Métrica':<20} {'Gemini FS':<15} {'OpenAI FT':<15} {'Delta':<10}")
print("-" * 60)
# Gemini results from notebook 07
gemini_acc = 0.462
gemini_f1m = 0.476
gemini_f1w = 0.475
print(f"{'Acurácia':<20} {gemini_acc:.1%}{'':9} {ft_acc:.1%}{'':9} {ft_acc-gemini_acc:+.1%}")
print(f"{'F1 Macro':<20} {gemini_f1m:.3f}{'':10} {ft_f1_macro:.3f}{'':10} {ft_f1_macro-gemini_f1m:+.3f}")
print(f"{'F1 Weighted':<20} {gemini_f1w:.3f}{'':10} {ft_f1_weighted:.3f}{'':10} {ft_f1_weighted-gemini_f1w:+.3f}")

In [ ]:
# Matriz de confusão
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Gemini few-shot (carregar resultados do notebook 07)
gemini_path = "data/llm_checkpoint_few_shot.csv"
if os.path.exists(gemini_path):
    gemini_results = pd.read_csv(gemini_path)
    gemini_eval = gemini_results[gemini_results["predicted"] != "ERROR"]
    cm_gemini = confusion_matrix(gemini_eval["true_label"], gemini_eval["predicted"], labels=VALID_CATEGORIES)
    sns.heatmap(cm_gemini, annot=True, fmt="d", cmap="Oranges",
                xticklabels=VALID_CATEGORIES, yticklabels=VALID_CATEGORIES, ax=axes[0])
    axes[0].set_title(f"Gemini Few-Shot (Acc: {gemini_acc:.1%})")
else:
    # Se não tiver o checkpoint, usar resultados salvos
    gemini_saved = "data/llm_classification_results.csv"
    if os.path.exists(gemini_saved):
        gemini_results = pd.read_csv(gemini_saved)
        gemini_eval = gemini_results[gemini_results["predicted"] != "ERROR"]
        cm_gemini = confusion_matrix(gemini_eval["true_label"], gemini_eval["predicted"], labels=VALID_CATEGORIES)
        sns.heatmap(cm_gemini, annot=True, fmt="d", cmap="Oranges",
                    xticklabels=VALID_CATEGORIES, yticklabels=VALID_CATEGORIES, ax=axes[0])
        axes[0].set_title(f"Gemini Few-Shot (Acc: {gemini_acc:.1%})")
    else:
        axes[0].text(0.5, 0.5, "Gemini results not found", ha="center")
        axes[0].set_title("Gemini Few-Shot (sem dados)")
axes[0].set_xlabel("Predito")
axes[0].set_ylabel("Real")
axes[0].tick_params(axis="x", rotation=45)

# OpenAI fine-tuned
cm_ft = confusion_matrix(ft_eval["true_label"], ft_eval["predicted"], labels=VALID_CATEGORIES)
sns.heatmap(cm_ft, annot=True, fmt="d", cmap="Blues",
            xticklabels=VALID_CATEGORIES, yticklabels=VALID_CATEGORIES, ax=axes[1])
axes[1].set_xlabel("Predito")
axes[1].set_ylabel("Real")
axes[1].set_title(f"OpenAI Fine-Tuned (Acc: {ft_acc:.1%})")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig("process-log/screenshots/p13_openai_vs_gemini_confusion.png", dpi=150, bbox_inches="tight")
plt.show()
print("Salvo: process-log/screenshots/p13_openai_vs_gemini_confusion.png")

In [ ]:
# F1 por categoria — comparação
fig, ax = plt.subplots(figsize=(14, 7))
x = np.arange(len(VALID_CATEGORIES))
width = 0.35

ft_cat_f1 = []
gemini_cat_f1 = []
for c in VALID_CATEGORIES:
    ft_sub = ft_eval[ft_eval["true_label"] == c]
    ft_correct = (ft_sub["predicted"] == c).sum()
    ft_pred_as_c = (ft_eval["predicted"] == c).sum()
    precision = ft_correct / ft_pred_as_c if ft_pred_as_c > 0 else 0
    recall = ft_correct / len(ft_sub) if len(ft_sub) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    ft_cat_f1.append(f1)

# Gemini F1 from notebook 07 (actual values)
gemini_f1_by_cat = {
    "Access": 0.45, "Administrative rights": 0.22, "HR Support": 0.47,
    "Hardware": 0.54, "Internal Project": 0.64, "Miscellaneous": 0.32,
    "Purchase": 0.77, "Storage": 0.39
}
gemini_cat_f1 = [gemini_f1_by_cat[c] for c in VALID_CATEGORIES]

ax.bar(x - width/2, gemini_cat_f1, width, label="Gemini Few-Shot", color="#f97316", alpha=0.8)
ax.bar(x + width/2, ft_cat_f1, width, label="OpenAI Fine-Tuned", color="#3b82f6", alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(VALID_CATEGORIES, rotation=45, ha="right")
ax.set_ylabel("F1 Score")
ax.set_title("F1 por Categoria: Gemini Few-Shot vs OpenAI Fine-Tuned")
ax.legend()
ax.axhline(y=0.5, color="gray", linestyle="--", alpha=0.5)
ax.axhline(y=0.7, color="green", linestyle="--", alpha=0.3, label="Meta 70%")
plt.tight_layout()
plt.savefig("process-log/screenshots/p13_f1_comparison_by_category.png", dpi=150, bbox_inches="tight")
plt.show()
print("Salvo: process-log/screenshots/p13_f1_comparison_by_category.png")

## Seção 7: Custo Real e Conclusões

In [ ]:
# Custo real do fine-tuning (do job)
job_final = client.fine_tuning.jobs.retrieve(job.id)

print("=" * 60)
print("CUSTO REAL")
print("=" * 60)
if hasattr(job_final, 'trained_tokens') and job_final.trained_tokens:
    train_cost = job_final.trained_tokens / 1_000_000 * 3.00
    print(f"Tokens treinados: {job_final.trained_tokens:,}")
    print(f"Custo treino: ~US${train_cost:.2f}")
else:
    print("Tokens treinados: não disponível no job (verificar dashboard OpenAI)")

# Custo de inferência (validação)
val_input_tokens = len(val_df) * 130  # ~130 tokens por mensagem
val_output_tokens = len(val_df) * 3
inference_cost = val_input_tokens / 1_000_000 * 0.30 + val_output_tokens / 1_000_000 * 1.20
print(f"Custo inferência ({len(val_df)} tickets): ~US${inference_cost:.4f}")

print(f"\n{'='*60}")
print("COMPARAÇÃO DE CUSTO")
print(f"{'='*60}")
print(f"{'Abordagem':<35} {'Custo':<15} {'Acurácia':<10}")
print("-" * 60)
print(f"{'Gemini ZS (20% amostra)':<35} {'~US$0.12':<15} {'40.9%':<10}")
print(f"{'Gemini FS (20% amostra)':<35} {'~US$1.42':<15} {'46.2%':<10}")
print(f"{'OpenAI FT treino (20% dataset)':<35} {'~US$9':<15} {'—':<10}")
print(f"{'OpenAI FT inferência (20% val)':<35} {f'~US${inference_cost:.2f}':<15} {f'{ft_acc:.1%}':<10}")

# Salvar resultados
results_df.to_csv("data/openai_ft_results.csv", index=False)
print(f"\nResultados salvos em data/openai_ft_results.csv")

# Limpar checkpoint
if os.path.exists(checkpoint_path):
    os.remove(checkpoint_path)
    print(f"Checkpoint removido")

print(f"\n{'='*60}")
print("CONCLUSÕES")
print(f"{'='*60}")
improvement = ft_acc - gemini_acc
print(f"1. Fine-tuning melhorou acurácia em {improvement:+.1%} vs Gemini few-shot")
if ft_acc >= 0.70:
    print(f"2. ✅ Atingiu Cenário B (≥70%) — viável para auto-roteamento parcial")
elif ft_acc >= 0.50:
    print(f"2. ⚠️ Entre Cenário A e B — melhor que few-shot, ainda precisa refinamento")
else:
    print(f"2. ❌ Abaixo de 50% — fine-tuning não foi suficiente")
print(f"3. Custo marginal de inferência: ~US$0.04/1000 tickets")
print(f"4. Modelo: {FINE_TUNED_MODEL}")